### Final Version of Data Preprocessing

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import chi2_contingency

In [67]:
# Load the dataset
df = pd.read_csv('../../../data/earthquakes_data.csv')

# Display sample data
print("Sample Data:")
print(df.sample(10))
print("\n" + "="*80 + "\n")

# Basic information
print("Dataset Information:")
df.info()
print("\n" + "="*80 + "\n")

# Dataset shape
print(f"Dataset Shape: {df.shape}")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print("\n" + "="*80 + "\n")

# Column names
print("Column Names:")
print(df.columns.tolist())
print("\n" + "="*80 + "\n")

Sample Data:
       Unnamed: 0                      time  latitude  longitude  depth   mag  \
16599       16599  1973-09-20T20:43:39.800Z     9.048    123.789  560.0  6.00   
47146       47146  1993-05-30T17:08:53.950Z     1.546    127.207   80.9  6.10   
30100       30100  1982-10-01T16:53:50.840Z    37.718    139.621  144.8  5.10   
14704       14704  1971-07-27T18:08:41.620Z    -5.651    151.498   40.0  5.64   
11428       11428  1965-03-30T00:21:00.010Z   -20.502   -173.701   20.0  5.91   
23935       23935  1978-04-02T02:24:03.400Z    -9.801    160.138   47.0  5.00   
7747         7747  1952-11-09T15:47:51.350Z    51.282    159.578   24.9  5.89   
64167       64167  2005-08-24T10:15:28.190Z    38.564    142.987   10.0  6.10   
60253       60253  2003-03-17T19:37:53.540Z    51.253    177.989   33.0  5.10   
12549       12549  1967-07-20T14:26:15.620Z    51.357    178.309   31.4  5.51   

      magType    nst   gap  dmin  ...                   updated  \
16599      mb    NaN   NaN  

In [68]:
print("Missing Value Analysis:")
print("-" * 80)

# Count missing values
missing_count = df.isnull().sum()
print("Missing Value Counts:")
print(missing_count[missing_count > 0])
print("\n")

# Calculate missing value percentage
missing_percent = round(df.isnull().sum() * 100 / len(df), 2)
print("Missing Value Percentage:")
print(missing_percent[missing_percent > 0])
print("\n" + "="*80 + "\n")

Missing Value Analysis:
--------------------------------------------------------------------------------
Missing Value Counts:
depth                285
nst                70641
gap                60350
dmin               80294
rms                28747
place                891
horizontalError    81674
depthError         49715
magError           67061
magNst             60025
dtype: int64


Missing Value Percentage:
depth               0.26
nst                64.14
gap                54.80
dmin               72.91
rms                26.10
place               0.81
horizontalError    74.16
depthError         45.14
magError           60.89
magNst             54.50
dtype: float64




In [69]:
print("Handling Missing Values:")
print("-" * 80)

# Step 1: Drop columns with >30% missing values
threshold = 30
cols_to_drop = missing_percent[missing_percent > threshold].index.tolist()
print(f"Columns dropped (>{threshold}% missing): {cols_to_drop}")
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
print(f"Shape after dropping columns: {df.shape}\n")

# Step 2: Drop ID-like and irrelevant columns
id_like_cols = ['Unnamed: 0', 'id', 'net', 'locationSource', 'magSource', 'place', 'updated']
existing_id_cols = [col for col in id_like_cols if col in df.columns]
print(f"Dropping ID-like/irrelevant columns: {existing_id_cols}")
df.drop(columns=existing_id_cols, inplace=True, errors='ignore')
print(f"Shape after dropping ID columns: {df.shape}\n")

# Step 3: Drop rows with missing 'depth' values
initial_rows = len(df)
df = df.dropna(subset=['depth'])
print(f"Rows dropped due to missing depth: {initial_rows - len(df)}")
print(f"Shape after dropping missing depth rows: {df.shape}\n")

# Step 4: Regression-based imputation for 'rms' column
if 'rms' in df.columns and df['rms'].isnull().sum() > 0:
    print("Applying regression-based imputation for 'rms'...")
    
    # Select predictor features
    predictors = ['mag', 'depth', 'nst', 'gap', 'dmin']
    available_features = [f for f in predictors if f in df.columns]
    
    # Separate known and missing RMS data
    df_known = df[df['rms'].notnull()].dropna(subset=available_features)
    df_missing = df[df['rms'].isnull()].dropna(subset=available_features)
    
    # Train and predict if sufficient data exists
    if not df_known.empty and not df_missing.empty:
        X_train = df_known[available_features]
        y_train = df_known['rms']
        
        model = LinearRegression()
        model.fit(X_train, y_train)
        
        # Impute missing values
        X_missing = df_missing[available_features]
        df.loc[df['rms'].isnull() & df.index.isin(df_missing.index), 'rms'] = model.predict(X_missing)
        
        print(f"RMS missing values after imputation: {df['rms'].isnull().sum()}")
    else:
        print("Insufficient data for regression imputation")
else:
    print("No 'rms' column or no missing values in 'rms'")

print("\n" + "="*80 + "\n")

# Check remaining missing values
print("Remaining Missing Values:")
remaining_missing = round(df.isnull().sum() * 100 / len(df), 2)
print(remaining_missing[remaining_missing > 0])
print("\n" + "="*80 + "\n")

Handling Missing Values:
--------------------------------------------------------------------------------
Columns dropped (>30% missing): ['nst', 'gap', 'dmin', 'horizontalError', 'depthError', 'magError', 'magNst']
Shape after dropping columns: (110133, 16)

Dropping ID-like/irrelevant columns: ['Unnamed: 0', 'id', 'net', 'locationSource', 'magSource', 'place', 'updated']
Shape after dropping ID columns: (110133, 9)

Rows dropped due to missing depth: 285
Shape after dropping missing depth rows: (109848, 9)

Applying regression-based imputation for 'rms'...
RMS missing values after imputation: 0


Remaining Missing Values:
Series([], dtype: float64)




In [70]:
# Feature Engineering

print("Feature Engineering:")
print("-" * 80)

# Step 1: Temporal feature extraction from 'time' column
if 'time' in df.columns:
    print("Extracting temporal features from 'time' column...")
    
    # Convert to datetime
    df['Datetime'] = pd.to_datetime(df['time'], errors='coerce')
    
    # Extract time components
    df['Year'] = df['Datetime'].dt.year
    df['Month'] = df['Datetime'].dt.month
    df['Day'] = df['Datetime'].dt.day
    df['Hour'] = df['Datetime'].dt.hour
    df['Minute'] = df['Datetime'].dt.minute
    df['Second'] = df['Datetime'].dt.second
    df['DayOfWeek'] = df['Datetime'].dt.dayofweek  # Monday=0, Sunday=6
    
    print("Temporal features created: Year, Month, Day, Hour, Minute, Second, DayOfWeek")
    
    # Drop original time columns
    df.drop(['time', 'Datetime'], axis=1, inplace=True)
    print("Dropped original 'time' and 'Datetime' columns\n")

# Step 2: Magnitude Type Conversion to Moment Magnitude (Mw)
print("Converting earthquake magnitudes to Moment Magnitude (Mw)...")
print("-" * 80)

def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5  # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    elif mag_type.lower() == 'ml':
        # Local Magnitude to Moment Magnitude
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type - return NaN
        return np.nan


# Apply the conversion to the DataFrame
if 'mag' in df.columns and 'magType' in df.columns:
    df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
    print(f"✓ Moment Magnitude (Mw) created from {df['magType'].nunique()} different magnitude types")
    print(f"\nMagnitude Type Distribution:")
    print(df['magType'].value_counts())
    print(f"\nSample Magnitude Conversions:")
    print(df[['mag', 'magType', 'Mw']].head(10))
    print()
else:
    print("Warning: 'mag' or 'magType' columns not found")
    print()

# Step 3: Earthquake Damage Potential Calculation (HAZUS Methodology)
print("Calculating earthquake damage potential using HAZUS methodology...")
print("-" * 80)

def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8  # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    # Return the final damage potential score
    return damage_potential


# Apply the conversion to the DataFrame
if 'Mw' in df.columns and 'depth' in df.columns:
    df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
    print(f"✓ Damage Potential Score calculated")
    print(f"\nDamage Potential Statistics:")
    print(df['damage_potential'].describe())
    print(f"\nSample Damage Potential Calculations:")
    print(df[['Mw', 'depth', 'damage_potential']].head(10))
    print()
else:
    print("Warning: 'Mw' or 'depth' columns not found")
    print()


print(f"Shape after additional feature engineering: {df.shape}")
print("\n" + "="*80 + "\n")



Feature Engineering:
--------------------------------------------------------------------------------
Extracting temporal features from 'time' column...
Temporal features created: Year, Month, Day, Hour, Minute, Second, DayOfWeek
Dropped original 'time' and 'Datetime' columns

Converting earthquake magnitudes to Moment Magnitude (Mw)...
--------------------------------------------------------------------------------
✓ Moment Magnitude (Mw) created from 25 different magnitude types

Magnitude Type Distribution:
magType
mb            45084
mw            24335
mwc           17458
mww           15652
ms             3157
mwb            3082
ml              485
mwr             451
md               50
mh               23
m                21
Mi               12
mwp               9
uk                8
fa                5
ml(texnet)        4
ms_20             3
mc                2
mlg               1
Md                1
Ml                1
mb_lg             1
lg                1
Mb              

In [71]:
# Feature Engineering
print("Feature Engineering:")
print("-" * 80)

# Step 1: Temporal feature extraction from 'time' column
if 'time' in df.columns:
    print("Extracting temporal features from 'time' column...")
    
    # Convert to datetime
    df['Datetime'] = pd.to_datetime(df['time'], errors='coerce')
    
    # Extract time components
    df['Year'] = df['Datetime'].dt.year
    df['Month'] = df['Datetime'].dt.month
    df['Day'] = df['Datetime'].dt.day
    df['Hour'] = df['Datetime'].dt.hour
    df['Minute'] = df['Datetime'].dt.minute
    df['Second'] = df['Datetime'].dt.second
    df['DayOfWeek'] = df['Datetime'].dt.dayofweek  # Monday=0, Sunday=6
    
    print("Temporal features created: Year, Month, Day, Hour, Minute, Second, DayOfWeek")
    
    # Drop original time columns
    df.drop(['time', 'Datetime'], axis=1, inplace=True)
    print("Dropped original 'time' and 'Datetime' columns\n")

# Step 2: MAGNITUDE TYPE CONVERSION TO MOMENT MAGNITUDE (Mw)

print("Converting earthquake magnitudes to Moment Magnitude (Mw)...")
print("-" * 80)

def convert_to_mw(mag_value, mag_type):
    """Convert various earthquake magnitude types to moment magnitude (Mw)."""
    # Reference conversions based on empirical relationships
    if mag_type.lower().startswith('mw'):
        # Moment Magnitude is already in the desired format
        return mag_value
    elif mag_type.lower() == 'ms':
        # Surface Wave Magnitude to Moment Magnitude
        return 1.05 * mag_value - 0.2
    elif mag_type.lower() == 'mb':
        # Body Wave Magnitude to Moment Magnitude
        if mag_value > 6.5:
            return 6.5 + (mag_value - 6.5) * 1.5  # Correction for saturation
        # For mb <= 6.5
        return 0.67 * mag_value + 3.2
    elif mag_type.lower() == 'ml':
        # Local Magnitude to Moment Magnitude
        return 1.2 * mag_value - 1.0
    else:
        # Unknown magnitude type - return NaN
        return np.nan


# Apply the conversion to the DataFrame
if 'mag' in df.columns and 'magType' in df.columns:
    df['Mw'] = df.apply(lambda row: convert_to_mw(row['mag'], row['magType']), axis=1)
    print(f"✓ Moment Magnitude (Mw) created from {df['magType'].nunique()} different magnitude types")
    print(f"\nMagnitude Type Distribution:")
    print(df['magType'].value_counts())
    print(f"\nSample Magnitude Conversions:")
    print(df[['mag', 'magType', 'Mw']].head(10))
    print()
else:
    print("Warning: 'mag' or 'magType' columns not found")
    print()

# Step 3: EARTHQUAKE DAMAGE POTENTIAL CALCULATION (HAZUS Methodology)

print("Calculating earthquake damage potential using HAZUS methodology...")
print("-" * 80)

def calculate_damage_potential_hazus(magnitude, depth):
    """Calculate earthquake damage potential using HAZUS methodology."""
    actual_depth = max(abs(depth), 1.0)  # Ensure depth is at least 1 km to avoid log(0)
    log_pga = magnitude - 3.5 * np.log10(actual_depth + 7) + 1.8  # HAZUS empirical formula
    pga = 10 ** log_pga  # Convert log10(PGA) to PGA in g
    # Calculate damage potential score (0 to 10 scale)
    # damage_potential = min(10.0, max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5))
    damage_potential = max(0.0, 2.5 * np.log10(pga + 0.01) + 7.5)
    # Return the final damage potential score
    return damage_potential


# Apply the conversion to the DataFrame
if 'Mw' in df.columns and 'depth' in df.columns:
    df['damage_potential'] = df.apply(lambda row: calculate_damage_potential_hazus(row['Mw'], row['depth']), axis=1)
    print(f"✓ Damage Potential Score calculated")
    print(f"\nDamage Potential Statistics:")
    print(df['damage_potential'].describe())
    print(f"\nSample Damage Potential Calculations:")
    print(df[['Mw', 'depth', 'damage_potential']].head(10))
    print()
else:
    print("Warning: 'Mw' or 'depth' columns not found")
    print()

# Shape after feature engineering
print(f"Shape after additional feature engineering: {df.shape}")
print("\n" + "="*80 + "\n")

# Step 4: URBANITY INDICATOR CREATION

gdf_points = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

# Load urban areas shapefile
urban_areas = gpd.read_file('../../../shp/ne_110m_admin_0_countries.shp')
urban_areas = urban_areas.to_crs(gdf_points.crs)

# Spatial join to check if points lie within urban areas
gdf_joined = gpd.sjoin(gdf_points, urban_areas, how='left', predicate='within')

# Create urbanity indicator: 1 if inside urban area, else 0
gdf_joined['urbanity_indicator'] = gdf_joined['index_right'].notnull().astype(int)

# Check distribution
print(gdf_joined['urbanity_indicator'].value_counts())

# Function to compute risk score based on urbanity
def compute_risk_score(row):
  mw = row['Mw']
  depth = max(row['depth'], 1) # Prevent division by zero
  urbanity = row['urbanity_indicator']

  if urbanity == 1: # Urban area
      # Amplified formula for urban areas: higher sensitivity
      risk_score = (mw**2) * (1/depth) * 1.5
  else: # Rural Area
      # Standard formula for rural area
      risk_score = mw * (1/depth)

  # Cap score to max of 10 for normalization
  return risk_score

# Apply to compute urban risk score
gdf_joined['risk_score'] = gdf_joined.apply(compute_risk_score, axis=1)

# Step 5: RISK SCORE CATEGORIZATION

# Function to assign risk score category based on score
def assign_risk_category(score):
  if score >= 7.5:
    return 'Very High'
  elif score >=5.0:
    return 'High'
  elif score >= 2.5:
    return 'Medium'
  else:
    return 'Low'

# Apply to assign risk category
gdf_joined['risk_category'] = gdf_joined['risk_score'].apply(assign_risk_category)

# Check results
print(gdf_joined[['Mw', 'depth', 'urbanity_indicator', 'risk_score', 'risk_category']].head(10))
print("\nRisk Category Distribution:")
print(gdf_joined['risk_category'].value_counts())

Feature Engineering:
--------------------------------------------------------------------------------
Converting earthquake magnitudes to Moment Magnitude (Mw)...
--------------------------------------------------------------------------------
✓ Moment Magnitude (Mw) created from 25 different magnitude types

Magnitude Type Distribution:
magType
mb            45084
mw            24335
mwc           17458
mww           15652
ms             3157
mwb            3082
ml              485
mwr             451
md               50
mh               23
m                21
Mi               12
mwp               9
uk                8
fa                5
ml(texnet)        4
ms_20             3
mc                2
mlg               1
Md                1
Ml                1
mb_lg             1
lg                1
Mb                1
ma                1
Name: count, dtype: int64

Sample Magnitude Conversions:
     mag magType    Mw
16  7.02      mw  7.02
17  6.84      mw  6.84
18  7.70      mw  7.70
19 

In [72]:
# Add features back to original DataFrame
df['urbanity_indicator'] = gdf_joined['urbanity_indicator']
df['risk_score'] = gdf_joined['risk_score']
df['risk_category'] = gdf_joined['risk_category']

In [73]:
df[['damage_potential', 'urbanity_indicator', 'risk_score', 'risk_category']].head()

,damage_potential,urbanity_indicator,risk_score,risk_category
16,17.803802,1,4.928040,Medium
17,17.353803,1,4.678560,Medium
18,17.528236,0,0.256667,Low
19,19.003802,0,0.500000,Low
20,17.978802,1,5.026810,High


In [74]:
df[['risk_score']].isnull().value_counts()

risk_score
False         109715
True             133
Name: count, dtype: int64

In [75]:
df[['risk_score']].describe()

,risk_score
count,109715.000000
mean,1.090893
std,4.596779
min,0.007580
25%,0.154545
50%,0.297691
75%,0.655000
max,85.617038


In [76]:
# descirbe 'risk_score' by grouping with 'urbanity_indicator'
df.groupby('urbanity_indicator')['risk_score'].describe()

,count,mean,std,min,25%,50%,75%,max
urbanity_indicator,,,,,,,,
0,85407.0,0.301648,0.315181,0.00758,0.125105,0.200515,0.5100,7.220000
1,24308.0,3.863933,9.227564,0.06360,0.850905,1.973768,3.9015,85.617038


In [77]:
# Outlier detection for 'risk_score' graph
df[['risk_score']].boxplot()
plt.title('Boxplot of Risk Score')

Text(0.5, 1.0, 'Boxplot of Risk Score')

In [78]:
# count outliers in 'risk_score'
Q1 = df['risk_score'].quantile(0.25)
Q3 = df['risk_score'].quantile(0.75)    
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df['risk_score'] < lower_bound) | (df['risk_score'] > upper_bound)]
print(f"Number of outliers in 'risk_score': {len(outliers)}")   

Number of outliers in 'risk_score': 15166


In [79]:
df.isnull().sum()

latitude                0
longitude               0
depth                   0
mag                     0
magType                 0
rms                     0
type                    0
status                  0
Year                    0
Month                   0
Day                     0
Hour                    0
Minute                  0
Second                  0
DayOfWeek               0
Mw                    133
damage_potential        0
urbanity_indicator      0
risk_score            133
risk_category           0
dtype: int64

In [80]:
# Drop nulls in 'risk_score' and 'Mw'
df.dropna(subset=['risk_score', 'Mw'], inplace=True)

In [82]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 109715 entries, 16 to 110132
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   latitude            109715 non-null  float64
 1   longitude           109715 non-null  float64
 2   depth               109715 non-null  float64
 3   mag                 109715 non-null  float64
 4   magType             109715 non-null  object 
 5   rms                 109715 non-null  float64
 6   type                109715 non-null  object 
 7   status              109715 non-null  object 
 8   Year                109715 non-null  int32  
 9   Month               109715 non-null  int32  
 10  Day                 109715 non-null  int32  
 11  Hour                109715 non-null  int32  
 12  Minute              109715 non-null  int32  
 13  Second              109715 non-null  int32  
 14  DayOfWeek           109715 non-null  int32  
 15  Mw                  109715 non-null  f

In [83]:
df.columns

Index(['latitude', 'longitude', 'depth', 'mag', 'magType', 'rms', 'type',
       'status', 'Year', 'Month', 'Day', 'Hour', 'Minute', 'Second',
       'DayOfWeek', 'Mw', 'damage_potential', 'urbanity_indicator',
       'risk_score', 'risk_category'],
      dtype='object')

In [84]:
# List of columns to drop
cols_to_drop = ['Year', 'Month', 'Day', 'Hour', 'Minute', 'Second', 'DayOfWeek', 'magType', 'type', 'status']

# Drop columns from the dataframe
df = df.drop(columns=cols_to_drop)

# Display remaining columns to verify
print(df.columns)

Index(['latitude', 'longitude', 'depth', 'mag', 'rms', 'Mw',
       'damage_potential', 'urbanity_indicator', 'risk_score',
       'risk_category'],
      dtype='object')


In [85]:
# Ordinal Encoding 'risk_category' 
risk_category_mapping = {
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Very High': 4
}
df['risk_category_encoded'] = df['risk_category'].map(risk_category_mapping)    

In [86]:
df.head()

,latitude,longitude,depth,mag,rms,Mw,damage_potential,urbanity_indicator,risk_score,risk_category,risk_category_encoded
16,41.758,23.249,15.0,7.02,1.032335,7.02,17.803802,1,4.928040,Medium,2
17,41.802,23.108,15.0,6.84,1.023418,6.84,17.353803,1,4.678560,Medium,2
18,52.763,160.277,30.0,7.70,1.065273,7.70,17.528236,0,0.256667,Low,1
19,51.424,161.638,15.0,7.50,1.056114,7.50,19.003802,0,0.500000,Low,1
20,30.684,100.608,15.0,7.09,1.035803,7.09,17.978802,1,5.026810,High,3


In [81]:
# # Min-Max scaling within each urbanity group
# df['risk_score_minmax_scaled'] = df.groupby('urbanity_indicator')['risk_score'].transform(
#     lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0
# )

# # Z-score standardization within each urbanity group
# def zscore_group(series):
#     scaler = StandardScaler()
#     values = series.values.reshape(-1, 1)
#     scaled_values = scaler.fit_transform(values).flatten()
#     return pd.Series(scaled_values, index=series.index)

# df['risk_score_zscore_scaled'] = df.groupby('urbanity_indicator')['risk_score'].transform(zscore_group)

# # Display first few rows to verify
# print(df[['urbanity_indicator', 'risk_score', 'risk_score_minmax_scaled', 'risk_score_zscore_scaled']].head())
